In [0]:
-- Databricks notebook source
-- ============================================================================
-- Gold layer: star schema for Happy Family Homestay
--
-- Source : happy_family_homestay.silver.homestay_clean
-- Load   : full refresh (CREATE OR REPLACE) on every run
--
-- Tables and grain:
--   dim_booker       one row per unique booker (guest_name + phone_number + email)
--   dim_calendar     one row per calendar date (view, always covers all bookings)
--   fact_booking     one row per booking
--   fact_daily_stay  one row per booking per stay night
--
-- Primary keys (deterministic: the same input always gives the same key, so
-- keys do not change between runs or when new bookings arrive):
--   dim_booker.booker_id         'BR-' + first 12 chars of md5(guest_name|phone_number|email)
--   dim_calendar.date            the date itself
--   fact_booking.booking_id      id from silver (natural key of the booking)
--   fact_daily_stay.daily_stay_id 'DS-' + booking_id + '-' + stay date (yyyyMMdd)
-- ============================================================================

-- COMMAND ----------

CREATE SCHEMA IF NOT EXISTS happy_family_homestay.gold
COMMENT 'Gold layer: star schema for analytics and dashboards';

-- COMMAND ----------

-- Booker dimension (1 row per unique booker)
-- The key is a hash of the natural key, so it is stable across runs.
-- COALESCE turns NULLs into '' so a missing value always hashes the same way.

CREATE OR REPLACE TABLE happy_family_homestay.gold.dim_booker AS
SELECT
    CONCAT(
        'BR-',
        SUBSTRING(
            md5(CONCAT_WS('|',
                COALESCE(guest_name, ''),
                COALESCE(phone_number, ''),
                COALESCE(email, '')
            )),
            1, 12
        )
    )                AS booker_id,
    guest_name,
    phone_number,
    email,
    MAX(city)        AS origin_city
FROM happy_family_homestay.silver.homestay_clean
GROUP BY
    guest_name,
    phone_number,
    email;

-- COMMAND ----------

-- Calendar dimension (1 row per date, from the first check-in to the last check-out)
-- A view, so it extends automatically when new bookings arrive.

CREATE OR REPLACE VIEW happy_family_homestay.gold.dim_calendar AS
WITH date_bounds AS (
    SELECT
        MIN(check_in)  AS min_date,
        MAX(check_out) AS max_date
    FROM happy_family_homestay.silver.homestay_clean
)
SELECT
    date,
    DATE_FORMAT(date, 'EEEE')      AS day_of_week,
    DAYOFWEEK(date) IN (1, 7)      AS is_weekend,   -- 1 = Sunday, 7 = Saturday
    DATE_FORMAT(date, 'yyyy-MM')   AS year_month
FROM date_bounds
CROSS JOIN LATERAL (
    SELECT EXPLODE(SEQUENCE(min_date, max_date, INTERVAL 1 DAY)) AS date
);

-- COMMAND ----------

-- Booking fact (1 row per booking)
-- booker_id is looked up from dim_booker on the same natural key used to build it.

CREATE OR REPLACE TABLE happy_family_homestay.gold.fact_booking AS
SELECT
    s.id                        AS booking_id,
    g.booker_id,
    s.created_at,
    s.check_in,
    s.check_out,
    s.stay_nights,
    s.guest_count,
    s.total_rooms_booked        AS rooms_booked,
    s.city                      AS origin_city,
    s.visiting_purpose,
    s.acquisition_channel,
    s.nightly_rate,
    s.total_revenue
FROM happy_family_homestay.silver.homestay_clean AS s
LEFT JOIN happy_family_homestay.gold.dim_booker AS g
    ON  COALESCE(s.guest_name,   '') = COALESCE(g.guest_name,   '')
    AND COALESCE(s.phone_number, '') = COALESCE(g.phone_number, '')
    AND COALESCE(s.email,        '') = COALESCE(g.email,        '');

-- COMMAND ----------

-- Daily stay fact (1 row per booking per stay night)
-- The check-out day is not a stay night, so the sequence ends the day before.
-- The key combines booking_id and the date, so it never changes between runs.

CREATE OR REPLACE TABLE happy_family_homestay.gold.fact_daily_stay AS
SELECT
    CONCAT('DS-', s.id, '-', DATE_FORMAT(d.stay_date, 'yyyyMMdd')) AS daily_stay_id,
    s.id                        AS booking_id,
    d.stay_date,
    s.guest_count,
    s.total_rooms_booked,
    s.nightly_rate              AS daily_revenue
FROM happy_family_homestay.silver.homestay_clean AS s
CROSS JOIN LATERAL (
    SELECT EXPLODE(SEQUENCE(s.check_in, DATE_SUB(s.check_out, 1))) AS stay_date
) AS d;